<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [13]:
import os
print(os.getcwd())

/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Answer

**Task Type:** Scoring (Priority Scoring)

My selected lane is **Refresh / Content Opportunity Scoring**.

The objective is to assign each content page a priority score based on observable search and content signals. Pages with higher scores should be reviewed or refreshed first.

This is a scoring problem because the goal is to rank pages according to their refresh priority rather than simply placing them into fixed categories.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [14]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the proxy target used in the starter notebooks
df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

df[["trend_direction", "is_declining_label"]].head(10)

,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


### Answer

**Target / Proxy:**

The target is whether a content page should be prioritized for refresh. In the starter dataset, a practical proxy is the **is_declining_label**, which is derived from the observed `trend_direction` of a page.

This proxy is based on observed search performance rather than manual opinions. It allows the model to learn patterns associated with declining pages so that future content can be prioritized for review.

### Answer

**Success Metric:**

The main success metric is **Precision@50**. This measures how many of the top 50 pages recommended for refresh are actually declining.

This metric is suitable because the goal is to prioritize the most important pages first. A higher Precision@50 means the model helps content teams spend their time reviewing pages that are more likely to need refreshing.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [18]:
import json

results = json.load(open("outputs/model_results.json"))

print("Baseline Precision@50:",
      results["baseline"]["baseline_precision_at_50"])

print("Random Forest Precision@50:",
      results["models"]["random_forest"]["precision_at_50"])

Baseline Precision@50: 0.24
Random Forest Precision@50: 0.74


### Answer

**Unit of Analysis:**

In this project, **one row represents one content page**.

Each row contains observable information about a single page, such as impressions, click-through rate (CTR), average search position, content age, and other search performance metrics.

The ML model evaluates each page independently and predicts whether it should be prioritized for refresh.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
# Load the starter dataset
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# Show a few example rows
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Answer

A fixed rule can only capture simple conditions, such as refreshing pages that have not been updated recently or have low click-through rates.

However, content performance depends on multiple factors working together, including impressions, average position, CTR, content age, and search trends.

Machine learning can learn these complex relationships from data and assign refresh priorities more effectively than a single hand-written rule.

The model provides decision support rather than replacing human judgment.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [11]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

print("Features used by the model:")
for feature in features:
    print("-", feature)

Features used by the model:
- content_age_days
- days_since_last_update
- impressions_90d
- avg_position
- ctr
- word_count


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [16]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
print(os.listdir())

Current working directory:
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter

Files and folders here:
['requirements.txt', 'outputs', 'data', '.github', '.gitignore', 'SETUP.md', '.git', 'docs', 'skills', 'README.md', 'submission', 'notebooks', 'AGENTS.md', 'work', 'CLAUDE.md', 'GUIDE.md', 'scripts', 'LICENSE', 'DATA_USE.md']


In [17]:
import sys

!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queu